In [ ]:
#use "lib/nb.ml";;

## Drawing Abstract Syntax Trees with GraphViz

An abstract syntax tree is a *nested tuple* of the type `nested`, which is defined in the file `../lib/nested.ml`:
its leaves are strings, numbers, and Boolean values, and an inner node is a tuple whose first component is the
label of the node, followed by the children.  For example, the tree `("+", ("NUMBER", "1"), "x")` has the root
`+` with the children `NUMBER` and `x`, and the node `NUMBER` has the child `1`.  If the first component of a
tuple is a tuple itself, as in `((":=", ";"), ("ID", "x"), e)`, this tuple is used as the label.

In [ ]:
load_lib "graphviz.ml"

`node_map`: A dictionary mapping a node's address (a list of ints) to its name (string).  Since this dictionary
is changed while the tree is traversed, we use a *hash table*, which is a mutable dictionary.

In [ ]:
type node_map = (int list, string) Hashtbl.t

The function `tuple2dot` takes a nested tuple `t` as its argument.  This nested tuple is interpreted as an
*abstract syntax tree*.   This tree is visualized using the module `Graphviz` from the file `../lib/graphviz.ml`.
The graph is displayed with `Graphviz.display`.

The functions `assign_numbers` and `create_nodes` are defined below.  Since `tuple2dot` calls them, the next
cell uses *forward references*: `!assign_numbers_ref` is the function that is currently stored in the box
`assign_numbers_ref`, and the box is filled once the function has been defined.

In [ ]:
let not_yet_defined _ = failwith "not yet defined"

let assign_numbers_ref : (int list -> nested -> node_map -> int -> int) ref = ref not_yet_defined
let create_nodes_ref   : (Graphviz.t -> int list -> nested -> node_map -> unit) ref = ref not_yet_defined

let tuple2dot (t : nested) : Graphviz.t =
  let dot = Graphviz.digraph "Abstract Syntax Tree" in
  let nodes_2_names : node_map = Hashtbl.create 16 in
  ignore (!assign_numbers_ref [] t nodes_2_names 0);
  !create_nodes_ref dot [] t nodes_2_names;
  dot

The function `children t` returns the children of the node `t`: a leaf has no children, and the children of a
tuple are all its components except the first one, which is the label.  The function `label t` returns the
label of `t`: the value of a leaf, or the first component of a tuple.

In [ ]:
let children (t : nested) : nested list =
  match t with
  | Leaf _ -> []
  | Node [] -> []
  | Node (_ :: rest) -> rest

let label (t : nested) : string =
  match t with
  | Leaf (Str s) -> s
  | Leaf v -> Nested.value_to_string v
  | Node [] -> "()"
  | Node (Leaf (Str s) :: _) -> s
  | Node (Leaf v :: _) -> Nested.value_to_string v
  | Node (head :: _) -> Nested.to_string head

The function `assign_numbers` takes four arguments:
- `address` is the address of the node `t`, i.e. the list of the positions of the children that lead from the root to `t`,
- `t` is a nested tuple that is interpreted as a tree,
- `nodes2Numbers` is a hash table,
- `n` is a natural number.

Given a tree `t` that is represented as a nested tuple, the function `assign_numbers` assigns a unique natural number
to every node of `t`.  This assignment is stored in the hash table `nodes2Numbers`. `n` is the first natural number
that is used.  The function returns the smallest natural number that is still unused.

In [ ]:
let rec assign_numbers (address : int list) (t : nested) (nodes2Numbers : node_map) (n : int) : int =
  Hashtbl.replace nodes2Numbers address (string_of_int n);
  let n = ref (n + 1) in
  List.iteri (fun j child -> n := assign_numbers (address @ [ j + 1 ]) child nodes2Numbers !n) (children t);
  !n
;;
assign_numbers_ref := assign_numbers

The function `create_nodes` takes four arguments:
- `dot` is a graph of type `Graphviz.t`,
- `a` is the address of the node `t`,
- `t` is an abstract syntax tree represented as a nested tuple.
- `nodes_2_names` is a hash table mapping nodes in `t` to unique names
   that can be used as node names in graphviz.

The function creates the nodes in `t` and connects them via directed edges so that `t` is represented as a tree.

In [ ]:
let rec create_nodes (dot : Graphviz.t) (a : int list) (t : nested) (nodes_2_names : node_map) : unit =
  let root = Hashtbl.find nodes_2_names a in
  Graphviz.node dot ~attrs:[ ("label", label t) ] root;
  List.iteri
    (fun j c ->
      let child = Hashtbl.find nodes_2_names (a @ [ j + 1 ]) in
      Graphviz.edge dot root child;
      create_nodes dot (a @ [ j + 1 ]) c nodes_2_names)
    (children t)
;;
create_nodes_ref := create_nodes